# FastPLMs Binder Design

This notebook mirrors the binder design tutorial using only FastPLMs-owned models and helpers. Heavy local runs should be executed on the Linux GPU workstation or sent to Modal. The notebook assumes `cookbook/tutorials/binder_design_fastplms.py` is available from the repository root.

## Install Notebook Helpers

The script owns the model code and CLI. These packages are only for notebook orchestration, tables, and structure viewing.

In [ ]:
! pip install modal pandas pyarrow py3dmol tqdm biopython

## Modal Setup

Deploy the FastPLMs binder app once, then reuse the named class for blocking or spawned jobs. Hugging Face authentication should be configured outside this notebook.

In [ ]:
! modal token info
! modal deploy cookbook/tutorials/binder_design_fastplms.py

In [ ]:
from itertools import product
import json
from pathlib import Path

import modal
import pandas as pd
import py3Dmol
from tqdm.contrib.concurrent import thread_map

from cookbook.tutorials.binder_design_fastplms import select_official_designs

APP_NAME = "fastplms-binder-design"
FastPLMsBinderDesign = modal.Cls.from_name(APP_NAME, "FastPLMsBinderDesignModal")

## Tiny Local Smoke Run

Use this only on a GPU workstation with the FastPLMs Docker or runtime environment available. The reduced step count is for checking wiring and outputs, not for design quality.

In [ ]:
! python cookbook/tutorials/binder_design_fastplms.py --backend local --target-name pd-l1 --binder-name minibinder --steps 2 --batch-size 1 --output-dir binder_design_fastplms_smoke

## Blocking Modal Run

This waits for the remote job and returns the same artifacts as the local CLI: best sequences, optimization trajectory, critic rows, and an official-style selection table. The selection score mirrors the paper and Biohub notebook: pI-filter minibinders, average hero-critic iPTM, optionally average scaling-critic distogram proxy, then combine those two components.

In [ ]:
app = FastPLMsBinderDesign(
    use_scaling_critics=False,
    kernel_backend=None,
    compile_model=False,
)

best_sequences, trajectory, rows = app.design.remote(
    target_name="pd-l1",
    binder_name="minibinder",
    seed=0,
    batch_size=1,
    steps=20,
    output_dir="binder_design_fastplms_modal",
)

results = pd.DataFrame(rows)
selection = select_official_designs(results)
best_sequences, selection.head(), results.drop(columns=["pdb", "cif"]).head()

## Spawned Modal Run

Spawned calls are useful for long designs and sweeps. Store the call ID so the job can be reattached later.

In [ ]:
future = app.design.spawn(
    target_name="ctla4",
    binder_name="minibinder",
    seed=1,
    batch_size=1,
    steps=50,
    output_dir="binder_design_fastplms_ctla4",
)

future.object_id

In [ ]:
! modal app logs fastplms-binder-design -f --function-call {future.object_id}

In [ ]:
best_sequences, trajectory, rows = modal.FunctionCall.gather(future)[0]
results = pd.DataFrame(rows)
results.sort_values("final_loss").head()

## Resumable Sweep Manifest

Each spawned call is recorded to disk immediately. If the notebook stops, recreate `modal.FunctionCall` objects from the saved IDs and gather them later.

In [ ]:
sweep_dir = Path("binder_design_fastplms_sweep")
sweep_dir.mkdir(parents=True, exist_ok=True)
manifest_path = sweep_dir / "manifest.json"

axes = {
    "target_name": ["pd-l1"],
    "binder_name": ["minibinder", "trastuzumab_framework_vhvl"],
    "seed": [0, 1],
}
jobs = [dict(zip(axes, values)) for values in product(*axes.values())]

records = []
for index, job in enumerate(jobs):
    call = app.design.spawn(
        **job,
        batch_size=1,
        steps=50,
        output_dir=str(sweep_dir / f"job_{index:03d}"),
    )
    records.append({"index": index, "call_id": call.object_id, **job})
    manifest_path.write_text(json.dumps(records, indent=2))

pd.DataFrame(records)

In [ ]:
records = json.loads(manifest_path.read_text())
calls = [modal.FunctionCall.from_id(record["call_id"]) for record in records]

def call_status(call):
    graph = call.get_call_graph()
    return graph[0].status.name

status_table = pd.DataFrame(records)
status_table["status"] = thread_map(call_status, calls)
status_table

In [ ]:
outputs = modal.FunctionCall.gather(*calls)
rows = []
for record, output in zip(records, outputs):
    best_sequences, trajectory, critic_rows = output
    for row in critic_rows:
        rows.append({**record, **row})

sweep_results = pd.DataFrame(rows)
sweep_results.to_parquet(sweep_dir / "results.parquet", index=False)
selection = select_official_designs(sweep_results)
selection.to_parquet(sweep_dir / "selection.parquet", index=False)
selection.head()

## Visualize A Design

Visualize a row for the top official selection. When available, use the full Cutoff2025 hero critic structure for consistency with the Biohub tutorial visualization.

In [ ]:
display_rows = sweep_results if "sweep_results" in globals() else results
selection_rows = selection if "selection" in globals() else select_official_designs(display_rows)
selected_sequence = selection_rows.iloc[0]["designed_sequence"]
matching_rows = display_rows[display_rows["designed_sequence"] == selected_sequence]
preferred_rows = matching_rows[
    matching_rows["critic_name"] == "ESMFold2-Experimental-Cutoff2025"
]
row = preferred_rows.iloc[0] if len(preferred_rows) else matching_rows.iloc[0]

view = py3Dmol.view(width=720, height=520)
view.addModel(row["pdb"], "pdb")
view.setStyle({"chain": "A"}, {"cartoon": {"color": "lightgray"}})
view.setStyle({"chain": "B"}, {"cartoon": {"color": "deepskyblue"}})
view.zoomTo()
view.show()

## Local CLI Reference

Use `--backend local` for workstation GPU runs and `--backend modal` for a deployed Modal class. The script writes `trajectory.jsonl`, `best_sequences.fasta`, `results.parquet`, per-critic structures, and final logits into the selected output directory.

In [ ]:
! python cookbook/tutorials/binder_design_fastplms.py --help